# v11 A (GPU 0): count-matched lesion and silencing on FINISHED models (no training)
In every layer n_max = size of the smaller group (self or task). Compared: n_max random self neurons vs n_max random task neurons (and half of that, plus look-alike task neurons at half).
- **lesion** = weights re-drawn as at initialisation; **silence** = neuron held at its average.
- per layer (one layer at a time) and all 8 layers at once; 2 draws; loss rise on 200 val pairs per language; chrF (100 FLORES devtest sentences) for all 8 layers.
- Models: distant phases 10, 11, 12, 40, 41, 42 · Hindi-only 10, 30, 45, 60 · mixed 10, 30, 45, 60 · related 10-15.
Results: `analysis/v11/static/` · progress: `tail -f logs/v11_A_static.log`

In [ ]:
import os, sys, glob, json, time, subprocess
os.environ["HF_HUB_DISABLE_XET"] = "1"
GPU = "0"; LOG = "v11_A_static"
REPO = "/workspace/shreshth"; os.chdir(REPO); sys.path.insert(0, REPO)
from IPython.display import Image, display
from huggingface_hub import HfApi, hf_hub_download
FILES, HF_REPO = [], None
for attempt in range(5):
    try:
        api = HfApi(); HF_REPO = api.whoami()["name"] + "/cml-lang-self-checkpoints"; FILES = api.list_repo_files(HF_REPO); break
    except Exception as e:
        print("HF retry", attempt, type(e).__name__); time.sleep(10)
if not FILES:
    print("could not list the HF repo: only checkpoints already on this machine will be used")

def have(run, ph):
    return bool(glob.glob(f"runs/{run}/ckpt/p{ph:03d}_*.pt"))

def get(run, phases):
    """download what is missing; returns the phases that are on this machine afterwards"""
    ok = []
    for ph in phases:
        if not have(run, ph):
            name = [f for f in FILES if f.startswith(f"{run}/ckpt/p{ph:03d}_")]
            if name:
                try:
                    hf_hub_download(HF_REPO, name[0], local_dir="runs")
                except Exception as e:
                    print("download failed:", run, ph, type(e).__name__, e)
        if have(run, ph):
            ok.append(ph)
        else:
            print("not available (not here, not on HF):", run, ph)
    return ok

def run(langset, *cmd):
    """one call of core_v11.py; output shown here AND appended to logs/<LOG>.log (watch with: tail -f logs/<LOG>.log)"""
    env = {**os.environ, "CML_LANGSET": langset}; os.makedirs("logs", exist_ok=True)
    p = subprocess.Popen([sys.executable, "-u", "core_v11.py", "--gpu", GPU, *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    with open(f"logs/{LOG}.log", "a") as log:
        for line in p.stdout:
            print(line, end=""); log.write(line); log.flush()
    p.wait(); print("exit code", p.returncode)

def summary(part, out=None):
    cmd = [sys.executable, "core_v11.py", "--summarize", "--part", part] + (["--out", out] if out else [])
    r = subprocess.run(cmd, capture_output=True, text=True); print(r.stdout[-12000:]); print(r.stderr[-3000:])

DIST, HIN, MIX, REL = "v4b_continual_s0", "v4b_only_hi_s0", "v6b_mixed_s0", "v8_indic_continual_s0"

In [ ]:
if os.path.exists("analysis/v11/static/static.csv"):
    os.remove("analysis/v11/static/static.csv")                    # this part has no resume: start clean
PLAN = [("distant", "distant", DIST, [10, 11, 12, 40, 41, 42], None), ("distant", "hindi", HIN, [10, 30, 45, 60], "hi"),
        ("distant", "mixed", MIX, [10, 30, 45, 60], None), ("indic", "related", REL, [10, 11, 12, 13, 14, 15], None)]
for langset, fam, r, phs, ev in PLAN:
    ok = get(r, phs)
    if ok:
        cmd = ["--part", "static", "--family", fam, "--model", f"{fam}={r}:" + ",".join(str(p) for p in ok)]
        run(langset, *(cmd + (["--eval_langs", ev] if ev else [])))

In [ ]:
summary("static")